In [39]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/mnt/lareaulab/reliscu/code")

from parse_gtf import *

In [40]:
psi = pd.read_csv(f"data/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_SJ_PSI.csv", index_col=0)

mod_eigs_df = pd.read_csv("data/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_Claude_marker_genes_enrichments_ctype_abundance.csv", index_col=0)

In [41]:
data_source = "yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_Claude_marker_genes_enrichments_ctype_abundance"

### Add gene names to PSI data

In [42]:
# Parse GTF attribute column
gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCm39/gencode.vM35.annotation.gtf"
gtf = gtf_parse(gtf_file)
gtf_subset = gtf.loc[gtf['feature'].isin(["gene"])]
attrs_df = gtf_subset["attribute"].apply(extract_attributes).apply(pd.Series)
gtf_parsed = pd.concat([gtf_subset.drop(columns=["attribute"]), attrs_df], axis=1)

In [43]:
# Get PSI and GTF data ready to merge on gene IDs
gtf_parsed['gene_id'] = gtf_parsed['gene_id'].str.split(".").str[0]
psi['gene_id'] = psi.index.str.split("_").str[0]
psi['exon_id'] = psi.index.values

In [44]:
psi_anno = pd.merge(gtf_parsed[['gene_id', 'gene_name']], psi, on="gene_id", how="right")
psi_anno = psi_anno.set_index("exon_id").rename_axis(None)
psi_anno = psi_anno.drop(columns=["gene_id"])

In [45]:
psi_anno

,gene_name,Sample1,Sample2,Sample3,Sample4,Sample5,Sample6,Sample7,Sample8,Sample9,...,Sample191,Sample192,Sample193,Sample194,Sample195,Sample196,Sample197,Sample198,Sample199,Sample200
ENSMUSG00000033845_ProteinCoding_1,Mrpl15,0.993051,0.993742,0.991429,0.990923,0.992510,0.993995,0.991052,0.993992,0.992033,...,0.991705,0.992281,0.991128,0.991474,0.990411,0.992481,0.992344,0.991485,0.991585,0.989809
ENSMUSG00000025903_ProteinCoding_1,Lypla1,0.998125,0.997347,0.996916,0.998937,0.994748,0.993386,0.996764,0.998224,0.998424,...,0.998408,0.999413,0.997840,0.996734,0.998484,0.998364,0.998624,0.998346,0.998623,0.999360
ENSMUSG00000025903_ProteinCoding_2,Lypla1,0.986045,0.986886,0.982486,0.980021,0.988957,0.981987,0.981869,0.989045,0.983026,...,0.986563,0.983836,0.986367,0.978473,0.981423,0.989912,0.987035,0.986580,0.985086,0.988292
ENSMUSG00000002459_ProteinCoding_1,Rgs20,0.989238,0.989849,0.983940,0.986087,0.983217,0.989082,0.985796,0.986845,0.989053,...,0.990198,0.984464,0.987350,0.986844,0.990998,0.974828,0.983984,0.988425,0.989681,0.987894
ENSMUSG00000033793_ProteinCoding_1,Atp6v1h,0.998637,0.998660,0.998095,0.998607,0.997913,0.998648,0.998156,0.998579,0.998150,...,0.998606,0.998411,0.998287,0.998020,0.998059,0.998389,0.998245,0.998480,0.998517,0.998705
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000068457_ProteinCoding_1,Uty,0.590205,0.566337,0.515085,0.581954,0.460589,0.484352,0.546904,0.511290,0.549000,...,0.573166,0.623269,0.547408,0.507886,0.524025,0.580229,0.556364,0.557309,0.434411,0.536232
ENSMUSG00000068457_ProteinCoding_2,Uty,0.996085,0.983782,0.987740,0.996151,0.990953,0.988979,0.999292,0.982738,0.991676,...,0.990527,0.995422,0.990164,0.992162,0.992000,0.990836,0.985215,0.990067,0.985860,0.996448
ENSMUSG00000068457_NMD_3,Uty,1.000000,1.000000,0.996865,1.000000,1.000000,0.982583,1.000000,1.000000,0.997135,...,1.000000,0.991163,1.000000,1.000000,1.000000,0.998893,1.000000,1.000000,0.998566,0.992037
ENSMUSG00000068457_NMD_4,Uty,0.000000,0.035576,0.017825,0.003177,0.000000,0.009457,0.000000,0.000000,0.014717,...,0.015118,0.041860,0.002329,0.029710,0.014309,0.000000,0.010697,0.002343,0.001467,0.004587


### Calc. corr between ME and exon PSI

In [52]:
psi_numeric_df =  psi_anno.iloc[:, 1:].T

In [53]:
mod_eigs_df = mod_eigs_df.reindex(psi_numeric_df.index)

In [ ]:
sum(mod_eigs_df.index == psi_numeric_df.index)

200

In [61]:
corr_df = pd.DataFrame(
    {col: psi_numeric_df.corrwith(mod_eigs_df[col]) for col in mod_eigs_df.columns}
)
corr_df.insert(0, "Gene", psi_anno["gene_name"].values)

In [62]:
corr_df.head()

,Gene,All_GABAergic,All_Glutamatergic,All_Neuronal,Sst
ENSMUSG00000033845_ProteinCoding_1,Mrpl15,0.051067,0.085158,0.108965,0.029950
ENSMUSG00000025903_ProteinCoding_1,Lypla1,0.102846,0.126794,0.251381,0.215143
ENSMUSG00000025903_ProteinCoding_2,Lypla1,0.194571,0.188413,0.340858,0.257801
ENSMUSG00000002459_ProteinCoding_1,Rgs20,-0.043807,0.128003,0.022849,-0.021816
ENSMUSG00000033793_ProteinCoding_1,Atp6v1h,-0.032974,0.034567,0.035666,0.041638


In [70]:
corr_df.sort_values("Sst", ascending=False).head(n=20)

,Gene,All_GABAergic,All_Glutamatergic,All_Neuronal,Sst
ENSMUSG00000026797_ProteinCoding_1,Stxbp1,0.570562,-0.406148,0.277284,0.547020
ENSMUSG00000009079_ProteinCoding_2,Ewsr1,0.398076,0.395864,0.766263,0.400578
ENSMUSG00000026825_ProteinCoding_1,Dnm1,0.587901,-0.531439,0.212782,0.385771
ENSMUSG00000022744_other_1,Cldnd1,0.367852,0.157754,0.644130,0.379775
ENSMUSG00000022744_ProteinCoding_2,Cldnd1,0.368917,0.154808,0.629620,0.371974
ENSMUSG00000041890_ProteinCoding_5,Git2,0.334847,0.410545,0.640618,0.360875
ENSMUSG00000022744_ProteinCoding_1,Cldnd1,0.343655,0.177187,0.638921,0.359552
ENSMUSG00000053768_ProteinCoding_1,Chchd3,0.295907,0.545923,0.816065,0.347678
ENSMUSG00000032463_ProteinCoding_1,Faim,0.274011,0.512994,0.752752,0.345951
ENSMUSG00000032076_ProteinCoding_4,Cadm1,0.333970,0.225856,0.548689,0.344554


In [72]:
corr_df.to_csv(f"data/corrs/{data_source}_PSI_vs_exon_corr.csv")